In [1]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold

In [2]:
X_train = pd.read_csv('../data/X_train.csv',index_col='ROW_ID')

y_train = pd.read_csv('../data/y_train.csv',index_col='ROW_ID')
total_data = pd.concat([X_train,y_train],axis=1)
print(total_data['target'])

ROW_ID
0         0.009210
1         0.000405
2         0.005967
3        -0.000542
4        -0.002579
            ...   
527068   -0.001769
527069    0.005640
527070    0.004016
527071    0.001617
527072    0.000556
Name: target, Length: 527073, dtype: float64


In [3]:
total_data['CROSS_SECTIONAL_AVG_RET1'] = (
    total_data.groupby('TS')['RET_1'].transform('mean')
)


# --- 1. distribution check ---

In [4]:
print(total_data['CROSS_SECTIONAL_AVG_RET1'].describe())
print(f"\nNaN count: {total_data['CROSS_SECTIONAL_AVG_RET1'].isna().sum()}")

count    527073.000000
mean          0.000079
std           0.000774
min          -0.005950
25%          -0.000354
50%           0.000071
75%           0.000466
max           0.005908
Name: CROSS_SECTIONAL_AVG_RET1, dtype: float64

NaN count: 0


# --- 2. how many allocations are actually present per TS? (small n per day would make
#     this average noisy and less trustworthy) ---

In [9]:
allocs_per_ts = total_data.groupby('TS')['ALLOCATION'].nunique()
print("\nAllocations per TS:")
print(allocs_per_ts.describe())


Allocations per TS:
count    2522.000000
mean      208.990087
std        81.903327
min        65.000000
25%       140.000000
50%       276.000000
75%       276.000000
max       276.000000
Name: ALLOCATION, dtype: float64


In [6]:
corr = total_data['CROSS_SECTIONAL_AVG_RET1'].corr(total_data['target'])
print(f"\nCROSS_SECTIONAL_AVG_RET1: Pearson corr with target = {corr:.4f}")


CROSS_SECTIONAL_AVG_RET1: Pearson corr with target = 0.0369


In [7]:

cs_sign = np.sign(total_data['CROSS_SECTIONAL_AVG_RET1'])
target_sign = np.sign(total_data['target'])
match_rate = (cs_sign == target_sign).mean() * 100
print(f"CROSS_SECTIONAL_AVG_RET1 sign-match rate with target = {match_rate:.2f}%")

CROSS_SECTIONAL_AVG_RET1 sign-match rate with target = 50.63%


In [8]:
ret1_sign = np.sign(total_data['RET_1'])
agrees_with_market = (ret1_sign == cs_sign)

match_agree = (ret1_sign[agrees_with_market] == target_sign[agrees_with_market]).mean() * 100
match_disagree = (ret1_sign[~agrees_with_market] == target_sign[~agrees_with_market]).mean() * 100
print(f"\nRET_1 sign-match when RET_1 AGREES with market direction: {match_agree:.2f}%")
print(f"RET_1 sign-match when RET_1 DISAGREES with market direction: {match_disagree:.2f}%")


RET_1 sign-match when RET_1 AGREES with market direction: 52.23%
RET_1 sign-match when RET_1 DISAGREES with market direction: 51.45%


In [10]:
print(f"\nCROSS_SECTIONAL_AVG_RET1: Pearson corr with target = {corr:.4f}")
print(f"CROSS_SECTIONAL_AVG_RET1 sign-match rate with target = {match_rate:.2f}%")


CROSS_SECTIONAL_AVG_RET1: Pearson corr with target = 0.0369
CROSS_SECTIONAL_AVG_RET1 sign-match rate with target = 50.63%
